# Занятие 38. Решение практики (только преподаватель)

**Не выдавать студентам.**

Код заполнен для проверки преподавателем. Главная модель — **RandomForestClassifier** (теория — занятие 37).

Решение сравнивает одно дерево, bagging и random forest; настраивает `n_estimators`, проверяет OOB и permutation importance.


---
## Дано: make_classification

20 признаков — как в теории.

In [ ]:
import numpy as np
from sklearn.datasets import make_classification

X, y = make_classification(
    n_samples=1000, n_features=20, n_informative=7, flip_y=0.08, random_state=42
)
print('Объектов:', len(X))

---
## Задание 0. Split (~8 мин)

Импорты: `train_test_split`, `DecisionTreeClassifier`, `BaggingClassifier`, `RandomForestClassifier`, `accuracy_score`, `permutation_importance`. Split 70/30, stratify, `RANDOM_STATE=42`.

**Критерий:** train/val готовы.

In [ ]:
RANDOM_STATE = 42
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import BaggingClassifier, RandomForestClassifier
from sklearn.metrics import accuracy_score
from sklearn.inspection import permutation_importance
import matplotlib.pyplot as plt

X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.3, stratify=y, random_state=RANDOM_STATE,
)
print(len(X_train), len(X_val))

---
## Задание 1. Три модели (~15 мин)

Одно дерево, bagging (150 estimators), random forest (150). Train и validation accuracy.

**Критерий:** bagging и/или RF заметно выше одного дерева на validation.

In [ ]:
models = {
    'tree': DecisionTreeClassifier(random_state=RANDOM_STATE),
    'bagging': BaggingClassifier(
        estimator=DecisionTreeClassifier(random_state=RANDOM_STATE),
        n_estimators=150, random_state=RANDOM_STATE, n_jobs=-1,
    ),
    'rf': RandomForestClassifier(n_estimators=150, random_state=RANDOM_STATE, n_jobs=-1),
}
for name, m in models.items():
    m.fit(X_train, y_train)
    print(name, accuracy_score(y_train, m.predict(X_train)),
          accuracy_score(y_val, m.predict(X_val)))

---
## Задание 2. n_estimators (~15 мин)

График validation accuracy vs `n_estimators` in `[1,5,20,60,150,300]`.

**Критерий:** кривая выходит на плато.

In [ ]:
ns = [1, 5, 20, 60, 150, 300]
val_scores = []
for n in ns:
    m = RandomForestClassifier(n_estimators=n, random_state=RANDOM_STATE, n_jobs=-1)
    m.fit(X_train, y_train)
    val_scores.append(accuracy_score(y_val, m.predict(X_val)))
plt.plot(ns, val_scores, 'o-')
plt.xlabel('n_estimators')
plt.ylabel('validation accuracy')
plt.title('Плато качества леса')
plt.show()

---
## Задание 3. OOB score (~10 мин)

`RandomForestClassifier(oob_score=True, n_estimators=300)`. Сравните OOB и validation.

**Критерий:** оба числа напечатаны.

In [ ]:
rf = RandomForestClassifier(
    n_estimators=300, oob_score=True, random_state=RANDOM_STATE, n_jobs=-1,
)
rf.fit(X_train, y_train)
print('OOB:', rf.oob_score_)
print('val:', accuracy_score(y_val, rf.predict(X_val)))

---
## Задание 4. max_depth (~12 мин)

`max_depth` in `[None, 5, 10, 20]` при `n_estimators=150`. Лучший по validation.

**Критерий:** `best_depth` выбран.

In [ ]:
best_depth, best_acc = None, -1
for d in [None, 5, 10, 20]:
    m = RandomForestClassifier(
        n_estimators=150, max_depth=d, random_state=RANDOM_STATE, n_jobs=-1,
    )
    m.fit(X_train, y_train)
    acc = accuracy_score(y_val, m.predict(X_val))
    print(d, acc)
    if acc > best_acc:
        best_acc, best_depth = acc, d
print('best_depth:', best_depth)

---
## Задание 5. Bootstrap вручную (~12 мин)

100 повторов bootstrap длины `len(X_train)`. Средняя доля уникальных.

**Критерий:** ~0.63 при большом n.

In [ ]:
rng = np.random.default_rng(RANDOM_STATE)
n = len(X_train)
fracs = [len(np.unique(rng.choice(n, n, replace=True))) / n for _ in range(100)]
print('средняя доля уникальных:', round(np.mean(fracs), 3))

---
## Задание 6. Permutation importance (~15 мин)

Top-5 признаков на validation.

**Критерий:** barh или таблица top-5.

In [ ]:
rf = RandomForestClassifier(n_estimators=150, random_state=RANDOM_STATE, n_jobs=-1)
rf.fit(X_train, y_train)
r = permutation_importance(rf, X_val, y_val, n_repeats=10, random_state=RANDOM_STATE)
imp = r.importances_mean
top = np.argsort(imp)[-5:]
for i in top[::-1]:
    print(f'f{i}', round(imp[i], 4))

---
## Задание 7. Train vs val gap (~8 мин)

В markdown: почему высокий train acc у леса не всегда переобучение?

**Критерий:** 2–3 предложения.

# Лес усредняет много деревьев: на train каждое дерево может подгоняться,
# но ансамбль стабильнее одного дерева. Большой разрыв train–val бывает,
# но добавление деревьев реже ухудшает validation — в отличие от одной глубокой модели.

---
## Задание 8. Итог (~5 мин)

Сравните bagging и RF в одном предложении каждый.

**Критерий:** markdown.

# Bagging: много деревьев на разных bootstrap-выборках, усреднение снижает разброс.
# RF: то же + случайный поднабор признаков в узлах → деревья разнообразнее.